# Orpheus Swiss German Fine-Tuning Notebook

This notebook fine-tunes an Orpheus checkpoint on your own paired speech/text data using LoRA.

What you need:
- A CSV manifest with at least: audio file path, transcript text, voice name
- Audio files (wav/flac/mp3)
- A GPU with enough VRAM for QLoRA training

Training target format:
- Text prompt: `voice: transcript`
- Audio target: SNAC tokens packed into Orpheus speech token layout

In [ ]:
%pip install -U torch torchaudio transformers datasets accelerate peft bitsandbytes librosa soundfile pandas tqdm snac sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.8/526.8 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 125.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 56.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found exi

## 1) Authenticate and Set Paths

Run this once per session if the model is gated/private.

In [ ]:
!hf auth login

A new version of huggingface_hub (1.7.2) is available! You are using version 1.7.1.
To update, run: pip install -U huggingface_hub


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? [y/N]: 
Token is valid (permission: write).
The token `Orpheus` has been saved to /root/.cache/huggingface/stored

In [ ]:
from pathlib import Path
import os
import torch

# Base model to fine-tune
BASE_MODEL = "canopylabs/3b-de-ft-research_release"

# Choose one: "colab" or "local_5070_ti"
# ENV_PROFILE = os.environ.get("ORPHEUS_ENV_PROFILE", "local_5070_ti")
ENV_PROFILE = "colab"

COMMON_CFG = {
    # Data loading mode: "manifest" or "speecht5_metadata"
    "DATASET_FORMAT": "speecht5_metadata",

    # Generic manifest mode
    "MANIFEST_CSV": "./data/swiss_manifest.csv",
    "AUDIO_ROOT": "./data/audio",

    # SpeechT5 metadata mode (audio|transcription|normalized)
    "SPEECHT5_METADATA_CSV": "./data/metadata_train.csv",
    "SPEECHT5_AUDIO_DIR": "./data/wavs",
    "SPEECHT5_TEXT_FIELD": "transcription",  # "transcription" or "normalized"
    "DEFAULT_VOICE": "swiss",

    # Canonical columns used by downstream pipeline
    "AUDIO_PATH_COL": "audio_path",
    "TEXT_COL": "text",
    "VOICE_COL": "voice",

    "TARGET_SR": 24000,
    "MAX_SEQ_LEN": 4096,
    "VAL_SPLIT": 0.05,
    "SEED": 42,
    "LORA_R": 32,
    "LORA_ALPHA": 64,
    "LORA_DROPOUT": 0.05,
}

PROFILE_CFG = {
    "colab": {
        "SPEECHT5_METADATA_CSV": "/content/drive/MyDrive/data/tts/data/metadata_train.csv",
        "SPEECHT5_AUDIO_DIR": "/content/drive/MyDrive/data/tts/data/wavs",
        "OUTPUT_DIR": "/content/outputs/orpheus-swiss-lora",
        "TOKEN_CACHE_DIR": "/content/cache/orpheus_tokens",
        "TRAIN_BATCH_SIZE": 1,
        "EVAL_BATCH_SIZE": 1,
        "GRAD_ACCUM_STEPS": 24,
        "LEARNING_RATE": 2e-4,
        "NUM_EPOCHS": 2,
        "WARMUP_RATIO": 0.03,
        "LOGGING_STEPS": 10,
        "SAVE_STEPS": 200,
        "EVAL_STEPS": 200,
        "USE_4BIT": True,
        "PREFERRED_DTYPE": "fp16",
    },
    "local_5070_ti": {
        "OUTPUT_DIR": "./outputs/orpheus-swiss-lora",
        "TOKEN_CACHE_DIR": "./cache/orpheus_tokens",
        "TRAIN_BATCH_SIZE": 2,
        "EVAL_BATCH_SIZE": 1,
        "GRAD_ACCUM_STEPS": 12,
        "LEARNING_RATE": 2e-4,
        "NUM_EPOCHS": 2,
        "WARMUP_RATIO": 0.03,
        "LOGGING_STEPS": 10,
        "SAVE_STEPS": 200,
        "EVAL_STEPS": 200,
        "USE_4BIT": True,
        "PREFERRED_DTYPE": "bf16",
    }
}

if ENV_PROFILE not in PROFILE_CFG:
    raise ValueError(f"Unknown ENV_PROFILE={ENV_PROFILE}. Valid: {list(PROFILE_CFG.keys())}")

cfg = {**COMMON_CFG, **PROFILE_CFG[ENV_PROFILE]}

DATASET_FORMAT = cfg["DATASET_FORMAT"]
MANIFEST_CSV = cfg["MANIFEST_CSV"]
AUDIO_ROOT = cfg["AUDIO_ROOT"]
SPEECHT5_METADATA_CSV = cfg["SPEECHT5_METADATA_CSV"]
SPEECHT5_AUDIO_DIR = cfg["SPEECHT5_AUDIO_DIR"]
SPEECHT5_TEXT_FIELD = cfg["SPEECHT5_TEXT_FIELD"]
DEFAULT_VOICE = cfg["DEFAULT_VOICE"]
OUTPUT_DIR = cfg["OUTPUT_DIR"]
TOKEN_CACHE_DIR = cfg["TOKEN_CACHE_DIR"]
AUDIO_PATH_COL = cfg["AUDIO_PATH_COL"]
TEXT_COL = cfg["TEXT_COL"]
VOICE_COL = cfg["VOICE_COL"]
TARGET_SR = cfg["TARGET_SR"]
MAX_SEQ_LEN = cfg["MAX_SEQ_LEN"]
VAL_SPLIT = cfg["VAL_SPLIT"]
SEED = cfg["SEED"]
LORA_R = cfg["LORA_R"]
LORA_ALPHA = cfg["LORA_ALPHA"]
LORA_DROPOUT = cfg["LORA_DROPOUT"]
TRAIN_BATCH_SIZE = cfg["TRAIN_BATCH_SIZE"]
EVAL_BATCH_SIZE = cfg["EVAL_BATCH_SIZE"]
GRAD_ACCUM_STEPS = cfg["GRAD_ACCUM_STEPS"]
LEARNING_RATE = cfg["LEARNING_RATE"]
NUM_EPOCHS = cfg["NUM_EPOCHS"]
WARMUP_RATIO = cfg["WARMUP_RATIO"]
LOGGING_STEPS = cfg["LOGGING_STEPS"]
SAVE_STEPS = cfg["SAVE_STEPS"]
EVAL_STEPS = cfg["EVAL_STEPS"]
USE_4BIT = cfg["USE_4BIT"]
PREFERRED_DTYPE = cfg["PREFERRED_DTYPE"]

if PREFERRED_DTYPE == "bf16":
    TRAIN_DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32
elif PREFERRED_DTYPE == "fp16":
    TRAIN_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
else:
    TRAIN_DTYPE = torch.float32

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(TOKEN_CACHE_DIR).mkdir(parents=True, exist_ok=True)
print("Active profile:", ENV_PROFILE)
print("Dataset format:", DATASET_FORMAT)
print("Configured output dir:", OUTPUT_DIR)
print("Configured token cache:", TOKEN_CACHE_DIR)
print("Using 4-bit loading:", USE_4BIT)
print("Preferred dtype:", PREFERRED_DTYPE)

Active profile: colab
Dataset format: speecht5_metadata
Configured output dir: /content/outputs/orpheus-swiss-lora
Configured token cache: /content/cache/orpheus_tokens
Using 4-bit loading: True
Preferred dtype: fp16


## 2) Load Data and Build Train/Validation Splits

This notebook supports two input layouts via `DATASET_FORMAT`:
- `manifest`: CSV already has `audio_path`, `text`, `voice` columns
- `speecht5_metadata`: SpeechT5-style `metadata_train.csv` with `audio|transcription|normalized` plus a wav folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import random
import pandas as pd

random.seed(SEED)

def resolve_audio_path(p: str) -> str:
    p = str(p)
    path_obj = Path(p)
    if path_obj.is_absolute():
        return str(path_obj)
    if AUDIO_ROOT:
        return str((Path(AUDIO_ROOT) / path_obj).resolve())
    return str(path_obj.resolve())

def build_df_from_manifest() -> pd.DataFrame:
    df = pd.read_csv(MANIFEST_CSV)
    required_cols = {AUDIO_PATH_COL, TEXT_COL, VOICE_COL}
    missing = required_cols.difference(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in manifest mode: {missing}")

    df = df.copy()
    df[AUDIO_PATH_COL] = df[AUDIO_PATH_COL].apply(resolve_audio_path)
    return df

def build_df_from_speecht5_metadata() -> pd.DataFrame:
    metadata_df = pd.read_csv(
        SPEECHT5_METADATA_CSV,
        delimiter="|",
        header=None,
        names=["audio", "transcription", "normalized"],
        dtype=str,
    ).fillna("")

    valid_text_fields = {"transcription", "normalized"}
    if SPEECHT5_TEXT_FIELD not in valid_text_fields:
        raise ValueError(
            f"SPEECHT5_TEXT_FIELD must be one of {valid_text_fields}, got: {SPEECHT5_TEXT_FIELD}"
        )

    def to_wav_path(stem: str) -> str:
        stem = str(stem).strip()
        return str((Path(SPEECHT5_AUDIO_DIR) / f"{stem}.wav").resolve())

    text_series = metadata_df[SPEECHT5_TEXT_FIELD].astype(str).str.strip()
    if SPEECHT5_TEXT_FIELD == "normalized":
        fallback = metadata_df["transcription"].astype(str).str.strip()
        text_series = text_series.where(text_series.str.len() > 0, fallback)

    df = pd.DataFrame(
        {
            AUDIO_PATH_COL: metadata_df["audio"].apply(to_wav_path),
            TEXT_COL: text_series,
            VOICE_COL: DEFAULT_VOICE,
        }
    )
    return df

if DATASET_FORMAT == "manifest":
    df = build_df_from_manifest()
elif DATASET_FORMAT == "speecht5_metadata":
    df = build_df_from_speecht5_metadata()
else:
    raise ValueError("DATASET_FORMAT must be 'manifest' or 'speecht5_metadata'.")

df = df[df[AUDIO_PATH_COL].apply(lambda p: Path(p).exists())].copy()
df = df[df[TEXT_COL].astype(str).str.len() > 0].copy()

if len(df) < 10:
    print("Warning: very small dataset. Consider at least 30-60 minutes of speech per voice.")

val_size = max(1, int(len(df) * VAL_SPLIT))
val_df = df.sample(n=val_size, random_state=SEED)
train_df = df.drop(val_df.index).reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Total rows: {len(df)}")
print(f"Train rows: {len(train_df)}")
print(f"Val rows:   {len(val_df)}")
display(train_df.head(3))

Total rows: 2715
Train rows: 2580
Val rows:   135


,audio_path,text,voice
0,/content/drive/.shortcut-targets-by-id/1jLigUi...,"Momentan isch er en ""Parasite"", em Sigerfelm v...",swiss
1,/content/drive/.shortcut-targets-by-id/1jLigUi...,De Boeing vom Typ 737-800 NG seg erst set 2016...,swiss
2,/content/drive/.shortcut-targets-by-id/1jLigUi...,Jetz isch die Pflegefachfrau setem Mäntig arbe...,swiss


## 3) Convert Audio to Orpheus Target Tokens

This step runs SNAC encoding and packs codec tokens into the same 7-token interleaving expected by Orpheus speech generation.

In [ ]:
import librosa
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from snac import SNAC

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# Special token IDs used by Orpheus
SOH_TOKEN = 128259
EOT_TOKEN = 128009
EOH_TOKEN = 128260
SOA_TOKEN = 128257
EOA_TOKEN = 128258
PAD_TOKEN = 128263
SPEECH_TOKEN_BASE = 128266

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").to(DEVICE).eval()

def load_audio_24k_mono(audio_path: str, target_sr: int = 24000) -> torch.Tensor:
    wav, _ = librosa.load(audio_path, sr=target_sr, mono=True)
    wav = torch.tensor(wav, dtype=torch.float32).unsqueeze(0)
    return wav

def extract_snac_layers(encoded):
    if isinstance(encoded, dict):
        for key in ["codes", "audio_codes", "tokens"]:
            if key in encoded:
                encoded = encoded[key]
                break
    if isinstance(encoded, tuple):
        encoded = list(encoded)
    if not isinstance(encoded, list):
        raise TypeError(f"Unexpected SNAC encode output type: {type(encoded)}")

    if len(encoded) == 1 and isinstance(encoded[0], (list, tuple)):
        encoded = list(encoded[0])

    if len(encoded) != 3:
        raise ValueError(f"Expected 3 SNAC layers, got {len(encoded)}")

    out = []
    for x in encoded:
        if not torch.is_tensor(x):
            x = torch.tensor(x)
        x = x.squeeze().long().cpu()
        out.append(x)
    return out

def pack_layers_to_orpheus_tokens(layer1, layer2, layer3):
    n = min(layer1.shape[0], layer2.shape[0] // 2, layer3.shape[0] // 4)
    layer1 = layer1[:n]
    layer2 = layer2[:2 * n]
    layer3 = layer3[:4 * n]

    packed = []
    for i in range(n):
        packed.append(int(layer1[i]))
        packed.append(int(layer2[2 * i]) + 4096)
        packed.append(int(layer3[4 * i]) + (2 * 4096))
        packed.append(int(layer3[4 * i + 1]) + (3 * 4096))
        packed.append(int(layer2[2 * i + 1]) + (4 * 4096))
        packed.append(int(layer3[4 * i + 2]) + (5 * 4096))
        packed.append(int(layer3[4 * i + 3]) + (6 * 4096))

    return [x + SPEECH_TOKEN_BASE for x in packed]

def build_prompt(text: str, voice: str) -> str:
    text = str(text).strip()
    voice = str(voice).strip()
    return f"{voice}: {text}" if voice else text

def build_example(row) -> dict:
    prompt = build_prompt(row[TEXT_COL], row[VOICE_COL])
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]

    wav = load_audio_24k_mono(row[AUDIO_PATH_COL], target_sr=TARGET_SR).to(DEVICE)
    with torch.no_grad():
        snac_encoded = snac_model.encode(wav)

    l1, l2, l3 = extract_snac_layers(snac_encoded)
    speech_ids = pack_layers_to_orpheus_tokens(l1, l2, l3)

    prefix = [SOH_TOKEN] + prompt_ids + [EOT_TOKEN, EOH_TOKEN]
    sequence = prefix + [SOA_TOKEN] + speech_ids + [EOA_TOKEN]

    if len(sequence) > MAX_SEQ_LEN:
        return {}

    labels = sequence.copy()
    # Mask text-conditioning tokens and keep loss only on speech generation tokens.
    labels[:len(prefix)] = [-100] * len(prefix)

    return {
        "input_ids": sequence,
        "labels": labels
    }

def preprocess_split(df_split, split_name: str):
    records = []
    dropped = 0
    for _, row in tqdm(df_split.iterrows(), total=len(df_split), desc=f"Preprocessing {split_name}"):
        ex = build_example(row)
        if not ex:
            dropped += 1
            continue
        records.append(ex)

    print(f"{split_name}: kept={len(records)}, dropped={dropped}")
    return records

Using device: cuda


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/22.8M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/300 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/79.5M [00:00<?, ?B/s]

In [ ]:
# SNAC expects [batch, channels, time]. This override fixes shape mismatches from 1D waveforms.
def load_audio_24k_mono(audio_path: str, target_sr: int = 24000) -> torch.Tensor:
    wav, _ = librosa.load(audio_path, sr=target_sr, mono=True)  # wav: [time]
    wav = torch.tensor(wav, dtype=torch.float32)

    if wav.dim() == 1:
        wav = wav.unsqueeze(0)  # [channels=1, time]
    if wav.dim() == 2:
        wav = wav.unsqueeze(0)  # [batch=1, channels=1, time]

    return wav

In [ ]:
from datasets import Dataset

train_records = preprocess_split(train_df, "train")
val_records = preprocess_split(val_df, "val")

if len(train_records) == 0:
    raise RuntimeError("No train records were created. Check data paths, audio files, and MAX_SEQ_LEN.")

train_ds = Dataset.from_list(train_records)
val_ds = Dataset.from_list(val_records) if len(val_records) else Dataset.from_list(train_records[:1])

train_cache = str(Path(TOKEN_CACHE_DIR) / "train_ds")
val_cache = str(Path(TOKEN_CACHE_DIR) / "val_ds")
train_ds.save_to_disk(train_cache)
val_ds.save_to_disk(val_cache)

print("Saved tokenized datasets to:")
print(train_cache)
print(val_cache)

Preprocessing train:   0%|          | 0/2580 [00:00<?, ?it/s]

train: kept=2580, dropped=0


Preprocessing val:   0%|          | 0/135 [00:00<?, ?it/s]

val: kept=135, dropped=0


Saving the dataset (0/1 shards):   0%|          | 0/2580 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/135 [00:00<?, ? examples/s]

Saved tokenized datasets to:
/content/cache/orpheus_tokens/train_ds
/content/cache/orpheus_tokens/val_ds


## 4) Fine-Tune with QLoRA

In [ ]:
from dataclasses import dataclass
import torch
from datasets import load_from_disk
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

train_ds = load_from_disk(str(Path(TOKEN_CACHE_DIR) / "train_ds"))
val_ds = load_from_disk(str(Path(TOKEN_CACHE_DIR) / "val_ds"))

@dataclass
class SpeechCollator:
    pad_token_id: int = PAD_TOKEN
    label_pad_id: int = -100

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        input_ids = []
        attention_mask = []
        labels = []

        for f in features:
            seq = f["input_ids"]
            lab = f["labels"]
            pad_len = max_len - len(seq)

            input_ids.append(seq + [self.pad_token_id] * pad_len)
            attention_mask.append([1] * len(seq) + [0] * pad_len)
            labels.append(lab + [self.label_pad_id] * pad_len)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

bnb_compute_dtype = TRAIN_DTYPE
bnb_config = BitsAndBytesConfig(
    load_in_4bit=torch.cuda.is_available() and USE_4BIT,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=bnb_compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=bnb_compute_dtype,
    quantization_config=bnb_config if (torch.cuda.is_available() and USE_4BIT) else None,
)

model.config.use_cache = False
if torch.cuda.is_available():
    model = prepare_model_for_kbit_training(model)

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    eval_strategy="steps",  # FIXED
    save_strategy="steps",
    bf16=torch.cuda.is_available() and TRAIN_DTYPE == torch.bfloat16,
    fp16=torch.cuda.is_available() and TRAIN_DTYPE == torch.float16,
    gradient_checkpointing=True,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    max_grad_norm=1.0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=SpeechCollator(),
)

trainer.train()
trainer.save_model(str(Path(OUTPUT_DIR) / "adapter"))
tokenizer.save_pretrained(str(Path(OUTPUT_DIR) / "adapter"))
print("Adapter saved to:", str(Path(OUTPUT_DIR) / "adapter"))

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

trainable params: 48,627,712 || all params: 3,349,494,784 || trainable%: 1.4518


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss
200,4.323592,4.362205


Adapter saved to: /content/outputs/orpheus-swiss-lora/adapter


## 5) Optional: Merge Adapter into a Standalone Model

Use this if you want a single merged checkpoint for inference. Requires enough VRAM/RAM.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

adapter_dir = str(Path(OUTPUT_DIR) / "adapter")
merged_dir = str(Path(OUTPUT_DIR) / "merged")

base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map="auto", torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
peft_model = PeftModel.from_pretrained(base, adapter_dir)
merged = peft_model.merge_and_unload()
merged.save_pretrained(merged_dir)
AutoTokenizer.from_pretrained(BASE_MODEL).save_pretrained(merged_dir)
print("Merged model saved to:", merged_dir)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to: /content/outputs/orpheus-swiss-lora/merged


## 6) Inference with Your Fine-Tuned Adapter

After training, load the adapter and then use the same generation and decoding steps as your existing inference notebook.

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

adapter_dir = str(Path(OUTPUT_DIR) / "adapter")

base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32)
if torch.cuda.is_available():
    base_model = base_model.to("cuda")

model = PeftModel.from_pretrained(base_model, adapter_dir)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model.eval()

print("Adapter loaded. Use this model in your generation cells.")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Adapter loaded. Use this model in your generation cells.


## Notes for Better Swiss German Results

- Keep transcripts close to what speakers actually say (dialect spelling consistency helps).
- Keep per-voice style tags stable (for example always `tara` or always `griffin`).
- Filter noisy clips and long silences before training.
- Start with 1-2 epochs; overtraining can hurt prosody and pronunciation.

## 7) Quick Inference (Adapter or Merged Model)

This section mirrors the standalone inference notebook and lets you test your fine-tuned model directly here.

Run these cells after training. By default it loads the adapter at `OUTPUT_DIR/adapter`.

In [ ]:
import torch
from pathlib import Path
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from snac import SNAC

# Inference config
USE_ADAPTER = True  # True -> load OUTPUT_DIR/adapter, False -> load OUTPUT_DIR/merged if available
INFER_MODEL_PATH = str(Path(OUTPUT_DIR) / ("adapter" if USE_ADAPTER else "merged"))
chosen_voice = "swiss"
prompts = [
    "Hoi zame, ich teste jetzt mis finetuned modell uf Schwiizerdütsch.",
    "Hesch du hüt am Morge öpper im Buero gseh?",
    "Hättisch doch öppis gseit!",
    "Wivil choschtet es Kafi im Restaurant i diim Heimatland im Vergliich zu de Schwiiz?",
    "Hesch au scho Problem gha, wil mer nöimed nid mit de Charte het chönne zahle? Verzell!",
    "isch",
    "au"
]

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else (torch.float16 if torch.cuda.is_available() else torch.float32)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").to(device).eval()

if USE_ADAPTER:
    base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=dtype)
    if torch.cuda.is_available():
        base_model = base_model.to(device)
    model = PeftModel.from_pretrained(base_model, INFER_MODEL_PATH)
else:
    model = AutoModelForCausalLM.from_pretrained(INFER_MODEL_PATH, torch_dtype=dtype)
    if torch.cuda.is_available():
        model = model.to(device)

model.eval()
print("Loaded model for inference from:", INFER_MODEL_PATH)

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loaded model for inference from: /content/outputs/orpheus-swiss-lora/adapter


In [ ]:
# Build Orpheus-formatted text prompts
formatted_prompts = [f"{chosen_voice}: {p}" for p in prompts]
all_input_ids = [tokenizer(p, return_tensors="pt").input_ids for p in formatted_prompts]

start_token = torch.tensor([[128259]], dtype=torch.int64)  # SOH
end_tokens = torch.tensor([[128009, 128260]], dtype=torch.int64)  # EOT, EOH

all_modified_input_ids = [torch.cat([start_token, ids, end_tokens], dim=1) for ids in all_input_ids]
max_length = max(t.shape[1] for t in all_modified_input_ids)

all_padded_tensors = []
all_attention_masks = []
for modified_input_ids in all_modified_input_ids:
    padding = max_length - modified_input_ids.shape[1]
    padded_tensor = torch.cat([torch.full((1, padding), 128263, dtype=torch.int64), modified_input_ids], dim=1)
    attention_mask = torch.cat([torch.zeros((1, padding), dtype=torch.int64), torch.ones((1, modified_input_ids.shape[1]), dtype=torch.int64)], dim=1)
    all_padded_tensors.append(padded_tensor)
    all_attention_masks.append(attention_mask)

input_ids = torch.cat(all_padded_tensors, dim=0).to(device)
attention_mask = torch.cat(all_attention_masks, dim=0).to(device)

with torch.no_grad():
    generated_ids = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=1200,
        do_sample=True,
        temperature=0.6,
        top_p=0.95,
        repetition_penalty=1.1,
        num_return_sequences=1,
        eos_token_id=128258,
    )

print("Generation complete.")

Setting `pad_token_id` to `eos_token_id`:128258 for open-end generation.


Generation complete.


In [ ]:
from IPython.display import display, Audio

# Parse generated sequence into Orpheus speech token stream
token_to_find = 128257  # SOA
token_to_remove = 128258  # EOA

token_indices = (generated_ids == token_to_find).nonzero(as_tuple=True)
if len(token_indices[1]) > 0:
    last_occurrence_idx = token_indices[1][-1].item()
    cropped_tensor = generated_ids[:, last_occurrence_idx + 1:]
else:
    cropped_tensor = generated_ids

processed_rows = [row[row != token_to_remove] for row in cropped_tensor]
code_lists = []

for row in processed_rows:
    row_length = row.size(0)
    new_length = (row_length // 7) * 7
    trimmed_row = row[:new_length]
    trimmed_row = [int(t) - 128266 for t in trimmed_row]
    code_lists.append(trimmed_row)

def redistribute_codes(code_list):
    layer_1 = []
    layer_2 = []
    layer_3 = []
    n = len(code_list) // 7
    for i in range(n):
        layer_1.append(code_list[7 * i])
        layer_2.append(code_list[7 * i + 1] - 4096)
        layer_3.append(code_list[7 * i + 2] - (2 * 4096))
        layer_3.append(code_list[7 * i + 3] - (3 * 4096))
        layer_2.append(code_list[7 * i + 4] - (4 * 4096))
        layer_3.append(code_list[7 * i + 5] - (5 * 4096))
        layer_3.append(code_list[7 * i + 6] - (6 * 4096))

    codes = [
        torch.tensor(layer_1, dtype=torch.long, device=device).unsqueeze(0),
        torch.tensor(layer_2, dtype=torch.long, device=device).unsqueeze(0),
        torch.tensor(layer_3, dtype=torch.long, device=device).unsqueeze(0),
    ]
    return snac_model.decode(codes)

my_samples = [redistribute_codes(code_list) for code_list in code_lists]

if len(formatted_prompts) != len(my_samples):
    raise RuntimeError("Number of prompts and generated samples do not match")

for i, samples in enumerate(my_samples):
    print(formatted_prompts[i])
    display(Audio(samples.detach().squeeze().to("cpu").numpy(), rate=24000))

swiss: Hoi zame, ich teste jetzt mis finetuned modell uf Schwiizerdütsch.


swiss: Hesch du hüt am Morge öpper im Buero gseh?


swiss: Hättisch doch öppis gseit!


swiss: Wivil choschtet es Kafi im Restaurant i diim Heimatland im Vergliich zu de Schwiiz?


swiss: Hesch au scho Problem gha, wil mer nöimed nid mit de Charte het chönne zahle? Verzell!


In [ ]:
from pathlib import Path
from huggingface_hub import HfApi

# Required: set your destination repo explicitly (example: "griffing52/orpheus-swiss-lora")
HF_REPO_ID = "griffing52/orpheus-swiss-german-lora"
HF_PRIVATE = False

# "adapter" (recommended) or "merged"
PUSH_TARGET = "adapter"

target_dir = Path(OUTPUT_DIR) / PUSH_TARGET
if not HF_REPO_ID.strip():
    raise ValueError("Set HF_REPO_ID before uploading, e.g. 'your-username/your-repo'.")
if not target_dir.exists():
    raise FileNotFoundError(f"Model directory not found: {target_dir}")

api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, repo_type="model", private=HF_PRIVATE, exist_ok=True)

api.upload_folder(
    folder_path=str(target_dir),
    repo_id=HF_REPO_ID,
    repo_type="model",
    commit_message=f"Upload {PUSH_TARGET} from Orpheus Swiss German fine-tune",
)

print(f"Uploaded {PUSH_TARGET} to https://huggingface.co/{HF_REPO_ID}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ra/adapter/tokenizer.json: 100%|##########| 22.8MB / 22.8MB            

  ...adapter_model.safetensors:   0%|          |  562kB /  195MB            

  ...adapter/training_args.bin:   9%|9         |   483B / 5.20kB            

Uploaded adapter to https://huggingface.co/griffing52/orpheus-swiss-german-lora


In [ ]:
from pathlib import Path
from datetime import datetime
import shutil

DRIVE_BACKUP_ROOT = Path('/content/drive/MyDrive/model_backups/orpheus_swiss')

# What to copy from OUTPUT_DIR
COPY_ADAPTER = True
COPY_MERGED = True
COPY_LATEST_CHECKPOINT = True  # copies the highest-step checkpoint-* folder if found
COPY_ALL_CHECKPOINTS = False   # set True to copy every checkpoint-* folder

output_dir = Path(OUTPUT_DIR)
if not output_dir.exists():
    raise FileNotFoundError(f'OUTPUT_DIR not found: {output_dir}')

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
backup_dir = DRIVE_BACKUP_ROOT / f'orpheus_swiss_backup_{timestamp}'
backup_dir.mkdir(parents=True, exist_ok=True)

def copy_if_exists(src: Path, dst_parent: Path):
    if src.exists():
        dst = dst_parent / src.name
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Copied: {src} -> {dst}')
    else:
        print(f'Skipped (not found): {src}')

if COPY_ADAPTER:
    copy_if_exists(output_dir / 'adapter', backup_dir)

if COPY_MERGED:
    copy_if_exists(output_dir / 'merged', backup_dir)

checkpoints = sorted([p for p in output_dir.glob('checkpoint-*') if p.is_dir()])
if COPY_ALL_CHECKPOINTS:
    for ckpt in checkpoints:
        copy_if_exists(ckpt, backup_dir)
elif COPY_LATEST_CHECKPOINT and checkpoints:
    copy_if_exists(checkpoints[-1], backup_dir)
elif COPY_LATEST_CHECKPOINT:
    print('No checkpoint-* directories found.')

# Save a minimal run manifest for reproducibility
manifest_path = backup_dir / 'backup_manifest.txt'
with manifest_path.open('w', encoding='utf-8') as f:
    f.write(f'BASE_MODEL={BASE_MODEL}\n')
    f.write(f'OUTPUT_DIR={output_dir}\n')
    f.write(f'BACKUP_DIR={backup_dir}\n')
    f.write(f'DATASET_FORMAT={DATASET_FORMAT}\n')
    f.write(f'MANIFEST_CSV={MANIFEST_CSV}\n')
    f.write(f'SPEECHT5_METADATA_CSV={SPEECHT5_METADATA_CSV}\n')
    f.write(f'SPEECHT5_AUDIO_DIR={SPEECHT5_AUDIO_DIR}\n')

print('\nBackup complete.')
print(f'Backup folder: {backup_dir}')

Copied: /content/outputs/orpheus-swiss-lora/adapter -> /content/drive/MyDrive/model_backups/orpheus_swiss/orpheus_swiss_backup_20260323_075138/adapter
Copied: /content/outputs/orpheus-swiss-lora/merged -> /content/drive/MyDrive/model_backups/orpheus_swiss/orpheus_swiss_backup_20260323_075138/merged
Copied: /content/outputs/orpheus-swiss-lora/checkpoint-216 -> /content/drive/MyDrive/model_backups/orpheus_swiss/orpheus_swiss_backup_20260323_075138/checkpoint-216

Backup complete.
Backup folder: /content/drive/MyDrive/model_backups/orpheus_swiss/orpheus_swiss_backup_20260323_075138
